# MedGemma base con dataset oficial

Este notebook usa el dataset actualizado en `dataset/`, que ya contiene imagenes, transcripciones expertas, labels, `locs_data`, mascaras cup/disc y splits fijos.

Objetivos:

1. Evaluar MedGemma base como clasificador glaucoma/non_glaucoma.
2. Evaluar MedGemma base como generador de descripcion contra `transcription` con BERTScore.
3. Correr ablation A/B/C1/C2/D1/D2 con mascaras reales.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"
REPO_NAME = "utils_medgemma"
MODEL_ID = "google/medgemma-1.5-4b-it"
SPLIT_FILE = "dataset/split_repetition_1.json"
MASK_KIND = "disc"  # opciones: "disc", "cup", "union"


## 1. Setup: Colab o cluster


In [ ]:
import os
import shutil
from pathlib import Path

try:
    import google.colab  # type: ignore
    IS_COLAB = True
except ModuleNotFoundError:
    IS_COLAB = False

ip = get_ipython()

if IS_COLAB:
    repo_dir = Path("/content") / REPO_NAME
    if repo_dir.exists():
        shutil.rmtree(repo_dir)
    ip.system(f'git clone "{REPO_URL}" "{repo_dir}"')
    ip.run_line_magic("cd", str(repo_dir))
    ip.system("pip install -q -r requirements.txt")
else:
    project_root = Path.cwd().resolve()
    if project_root.name == "notebooks":
        project_root = project_root.parent
    if not (project_root / "dataset").exists():
        raise RuntimeError("No encuentro dataset/. En Kabre corre el notebook desde la raiz del repo o ajusta project_root.")
    os.chdir(project_root)
    print(f"Ejecutando desde repo local: {project_root}")


## 2. Imports, token y carga de dataset

In [ ]:
import json
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

import numpy as np
import torch
from PIL import Image
from huggingface_hub import HfApi
from transformers import AutoProcessor, AutoModelForImageTextToText
from bert_score import score as bertscore

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if IS_COLAB and not HF_TOKEN:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("Define HF_TOKEN en el entorno. En Kabre: export HF_TOKEN=...; en Colab: guardalo en Secrets.")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DEVICE = "cuda"
DTYPE = torch.bfloat16

api = HfApi()
print(api.whoami(token=HF_TOKEN))
print(api.model_info(MODEL_ID, token=HF_TOKEN).modelId)
assert torch.cuda.is_available(), "Se requiere GPU CUDA para correr MedGemma."
print(torch.cuda.get_device_name(0))


In [ ]:
DATASET_ROOT = Path("dataset")

with open(SPLIT_FILE, "r", encoding="utf-8") as f:
    split_data = json.load(f)

def read_annotation(annotation_path):
    with open(DATASET_ROOT / annotation_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data[0] if isinstance(data, list) else data

def make_row(split_name, item):
    ann = read_annotation(item["annotation"])
    conditions = ann.get("locs_data", {}).get("conditions", []) or []
    is_glaucoma = "glaucoma" in [c.lower() for c in conditions]
    return {
        "split": split_name,
        "image": str(DATASET_ROOT / item["image"]),
        "mask_cup_npy": str(DATASET_ROOT / item["mask_cup_npy"]),
        "mask_disc_npy": str(DATASET_ROOT / item["mask_disc_npy"]),
        "annotation": str(DATASET_ROOT / item["annotation"]),
        "image_filename": ann.get("image_filename"),
        "label": ann.get("label"),
        "conditions": conditions,
        "target_label": "glaucoma" if is_glaucoma else "non_glaucoma",
        "transcription": ann.get("transcription", ""),
        "locs_data": ann.get("locs_data", {}),
    }

rows = []
for split_name in ["train", "validation", "test"]:
    rows.extend(make_row(split_name, item) for item in split_data[split_name])

train_rows = [r for r in rows if r["split"] == "train"]
val_rows = [r for r in rows if r["split"] == "validation"]
test_rows = [r for r in rows if r["split"] == "test"]

print("splits:", Counter(r["split"] for r in rows))
print("labels:", Counter(r["label"] for r in rows))
print("target:", Counter(r["target_label"] for r in rows))
print("empty transcription:", sum(not r["transcription"] for r in rows))

test_rows[0]

## 3. Cargar MedGemma base

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    token=HF_TOKEN,
).to(DEVICE)
model.eval()
print(type(model))

## 4. Helpers de generación, máscaras y métricas

In [ ]:
def build_messages(prompt, image_path=None):
    content = []
    if image_path is not None:
        image = Image.open(image_path).convert("RGB")
        content.append({"type": "image", "image": image})
    content.append({"type": "text", "text": prompt})
    return [{"role": "user", "content": content}]

def generate_medgemma(prompt, image_path=None, max_new_tokens=256):
    messages = build_messages(prompt, image_path)
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE)
    input_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0, input_len:]
    return processor.decode(generated, skip_special_tokens=True).strip()

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def load_mask(row, mask_kind="disc"):
    cup = np.load(row["mask_cup_npy"]).astype(bool)
    disc = np.load(row["mask_disc_npy"]).astype(bool)
    if mask_kind == "cup":
        return cup
    if mask_kind == "disc":
        return disc
    if mask_kind == "union":
        return cup | disc
    raise ValueError(f"Unknown mask_kind: {mask_kind}")

def make_overlay(image_path, mask, alpha=0.4, out_path="results/overlay.png"):
    image = np.asarray(Image.open(image_path).convert("RGB")).astype(np.float32)
    if mask.shape != image.shape[:2]:
        raise ValueError(f"mask shape {mask.shape} != image shape {image.shape[:2]}")
    red = np.array([255, 0, 0], dtype=np.float32)
    image[mask] = image[mask] * (1 - alpha) + red * alpha
    out = np.rint(image).clip(0, 255).astype(np.uint8)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(out).save(out_path)
    return out_path

def parse_glaucoma_label(text):
    clean = text.lower().strip()
    if clean in {"glaucoma", "non_glaucoma", "non glaucoma", "normal"}:
        return "non_glaucoma" if clean in {"non_glaucoma", "non glaucoma", "normal"} else "glaucoma"
    if re.search(r"non[-_\s]?glaucoma|no glaucoma|not glaucoma|normal", clean):
        return "non_glaucoma"
    if re.search(r"\bglaucoma\b|glaucomatous", clean):
        return "glaucoma"
    return "unknown"

def compute_binary_metrics(results):
    tp = sum(r["true_label"] == "glaucoma" and r["pred_label"] == "glaucoma" for r in results)
    tn = sum(r["true_label"] == "non_glaucoma" and r["pred_label"] == "non_glaucoma" for r in results)
    fp = sum(r["true_label"] == "non_glaucoma" and r["pred_label"] == "glaucoma" for r in results)
    fn = sum(r["true_label"] == "glaucoma" and r["pred_label"] == "non_glaucoma" for r in results)
    unknown = sum(r["pred_label"] == "unknown" for r in results)
    total = len(results)
    known = total - unknown
    return {
        "total": total,
        "known_predictions": known,
        "unknown": unknown,
        "accuracy_known_only": (tp + tn) / known if known else None,
        "accuracy_with_unknown_as_wrong": (tp + tn) / total if total else None,
        "sensitivity_glaucoma": tp / (tp + fn) if (tp + fn) else None,
        "specificity_non_glaucoma": tn / (tn + fp) if (tn + fp) else None,
        "confusion": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
    }

## 5. Clasificación zero-shot en test

In [ ]:
CLASSIFICATION_PROMPT = """
You are evaluating a color fundus photograph for glaucoma screening.
Choose exactly one label from this list:
- glaucoma
- non_glaucoma

Reply with only the label and no explanation.
""".strip()

classification_results = []
for idx, row in enumerate(test_rows, start=1):
    print(f"[{idx}/{len(test_rows)}]", row["target_label"], row["image"])
    raw_text = generate_medgemma(CLASSIFICATION_PROMPT, image_path=row["image"], max_new_tokens=32)
    pred_label = parse_glaucoma_label(raw_text)
    item = {
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "true_label": row["target_label"],
        "raw_text": raw_text,
        "pred_label": pred_label,
        "correct": pred_label == row["target_label"],
    }
    classification_results.append(item)
    print("raw:", raw_text, "pred:", pred_label, "correct:", item["correct"])

classification_metrics = compute_binary_metrics(classification_results)
save_json("results/official_base_classification_test.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "split": "test",
    "prompt": CLASSIFICATION_PROMPT,
    "metrics": classification_metrics,
    "results": classification_results,
})
classification_metrics

## 6. Descripción base + BERTScore en test

In [ ]:
DESCRIPTION_PROMPT = "Describe the ophthalmological findings in this fundus image."

description_outputs = []
for idx, row in enumerate(test_rows, start=1):
    print(f"[{idx}/{len(test_rows)}]", row["image"])
    generated = generate_medgemma(DESCRIPTION_PROMPT, image_path=row["image"], max_new_tokens=384)
    item = {
        "image": row["image"],
        "label": row["label"],
        "conditions": row["conditions"],
        "reference": row["transcription"],
        "generated": generated,
    }
    description_outputs.append(item)
    print("REF:", row["transcription"][:350])
    print("GEN:", generated[:350])

candidates = [x["generated"] for x in description_outputs]
references = [x["reference"] for x in description_outputs]
P, R, F1 = bertscore(candidates, references, lang="en", rescale_with_baseline=True, verbose=True)

for item, p, r, f in zip(description_outputs, P, R, F1):
    item["bertscore_precision"] = float(p)
    item["bertscore_recall"] = float(r)
    item["bertscore_f1"] = float(f)

description_summary = {
    "count": len(description_outputs),
    "bertscore_precision_mean": float(P.mean()),
    "bertscore_recall_mean": float(R.mean()),
    "bertscore_f1_mean": float(F1.mean()),
}

save_json("results/official_base_description_test_bertscore.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "split": "test",
    "prompt": DESCRIPTION_PROMPT,
    "summary": description_summary,
    "results": description_outputs,
})
description_summary

## 7. Ablation con máscara real

Usamos `MASK_KIND = disc` por defecto. También puedes probar `cup` o `union`.

In [ ]:
def format_distribution(distribution):
    return ", ".join(
        f"{label} ({prob:.0%})"
        for label, prob in sorted(distribution.items(), key=lambda item: item[1], reverse=True)
    )

def ablation_prompts(row, overlay_path, prediction="glaucoma", distribution=None):
    distribution = distribution or {"glaucoma": 0.92, "non_glaucoma": 0.08}
    dist = format_distribution(distribution)
    return {
        "A": (row["image"], "Describe the ophthalmological findings in this fundus image."),
        "B": (overlay_path, "The region highlighted in red was identified by an automatic segmentation system. Describe the ophthalmological findings, focusing on the highlighted region."),
        "C1": (row["image"], f"An ophthalmological classifier identifies the primary finding in this fundus image as: {prediction}. Describe the ophthalmological findings."),
        "C2": (row["image"], f"An ophthalmological classifier analyzed this fundus image and estimates: {dist}. Describe the ophthalmological findings."),
        "D1": (overlay_path, f"An ophthalmological classifier identifies the primary finding as: {prediction}. The region highlighted in red indicates the area where this finding is located. Describe the findings focusing on the highlighted region."),
        "D2": (overlay_path, f"An ophthalmological classifier estimates: {dist}. The region highlighted in red indicates the area where the main finding is located. Describe the findings in detail, focusing on the highlighted region and its relationship with the suggested diagnosis."),
    }

# Para no gastar demasiado, hacemos ablation en 3 muestras del test.
ablation_rows = test_rows[:3]
ablation_results = []
for row in ablation_rows:
    mask = load_mask(row, MASK_KIND)
    overlay_path = make_overlay(
        row["image"],
        mask,
        out_path=f"results/overlays/{Path(row['image']).stem}_{MASK_KIND}.png",
    )
    for condition, (image_path, prompt) in ablation_prompts(row, overlay_path).items():
        print("Running", Path(row["image"]).name, condition)
        text = generate_medgemma(prompt, image_path=image_path, max_new_tokens=384)
        ablation_results.append({
            "image": row["image"],
            "condition": condition,
            "mask_kind": MASK_KIND,
            "prompt_used": prompt,
            "reference": row["transcription"],
            "text": text,
        })
        print(text[:500])

save_json("results/official_base_ablation_real_mask.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "split": "test_subset",
    "mask_kind": MASK_KIND,
    "results": ablation_results,
})
len(ablation_results)